In [0]:
# DATA INGESTION (Bronze Layer)
df = spark.read.csv("/Workspace/Users/poojaoswal1628@gmail.com/customer_churn_dataset.csv",
                    header=True,
                    inferSchema=True)
df.show(5)

+----------+---+------+------+---------------+-------------+-------------+-----------------+---------------+-----------+----------------+-----+
|CustomerID|Age|Gender|Tenure|Usage Frequency|Support Calls|Payment Delay|Subscription Type|Contract Length|Total Spend|Last Interaction|Churn|
+----------+---+------+------+---------------+-------------+-------------+-----------------+---------------+-----------+----------------+-----+
|         2| 30|Female|    39|             14|            5|           18|         Standard|         Annual|      932.0|              17|    1|
|         3| 65|Female|    49|              1|           10|            8|            Basic|        Monthly|      557.0|               6|    1|
|         4| 55|Female|    14|              4|            6|           18|            Basic|      Quarterly|      185.0|               3|    1|
|         5| 58|  Male|    38|             21|            7|            7|         Standard|        Monthly|      396.0|              29

In [0]:
df.printSchema()

root
 |-- CustomerID: integer (nullable = true)
 |-- Age: integer (nullable = true)
 |-- Gender: string (nullable = true)
 |-- Tenure: integer (nullable = true)
 |-- Usage Frequency: integer (nullable = true)
 |-- Support Calls: integer (nullable = true)
 |-- Payment Delay: integer (nullable = true)
 |-- Subscription Type: string (nullable = true)
 |-- Contract Length: string (nullable = true)
 |-- Total Spend: double (nullable = true)
 |-- Last Interaction: integer (nullable = true)
 |-- Churn: integer (nullable = true)



In [0]:
# remove nulls
df= df.dropna()
df.show()

+----------+---+------+------+---------------+-------------+-------------+-----------------+---------------+-----------+----------------+-----+
|CustomerID|Age|Gender|Tenure|Usage Frequency|Support Calls|Payment Delay|Subscription Type|Contract Length|Total Spend|Last Interaction|Churn|
+----------+---+------+------+---------------+-------------+-------------+-----------------+---------------+-----------+----------------+-----+
|         2| 30|Female|    39|             14|            5|           18|         Standard|         Annual|      932.0|              17|    1|
|         3| 65|Female|    49|              1|           10|            8|            Basic|        Monthly|      557.0|               6|    1|
|         4| 55|Female|    14|              4|            6|           18|            Basic|      Quarterly|      185.0|               3|    1|
|         5| 58|  Male|    38|             21|            7|            7|         Standard|        Monthly|      396.0|              29

In [0]:
# save as Bronze 
df_raw = df
df_raw.write.format("delta").mode("overwrite").option("delta.columnMapping.mode","name").saveAsTable("churn_broze")

In [0]:
# data cleaning (silver layer)
df = spark.read.table("churn_broze")
df.printSchema()

root
 |-- CustomerID: integer (nullable = true)
 |-- Age: integer (nullable = true)
 |-- Gender: string (nullable = true)
 |-- Tenure: integer (nullable = true)
 |-- Usage Frequency: integer (nullable = true)
 |-- Support Calls: integer (nullable = true)
 |-- Payment Delay: integer (nullable = true)
 |-- Subscription Type: string (nullable = true)
 |-- Contract Length: string (nullable = true)
 |-- Total Spend: double (nullable = true)
 |-- Last Interaction: integer (nullable = true)
 |-- Churn: integer (nullable = true)



In [0]:
# file data types
from pyspark.sql.functions import col
df = df.withColumn("Total Spend", col("Total spend").cast("double"))

In [0]:
# handle nulls
df = df.dropna()

In [0]:
# feature engineering - rename columns
df = df.toDF(*[c.replace(' ', '_') .replace("(", "").replace(")", "") for
c in df.columns])

In [0]:
# feature engineering
from pyspark.sql.functions import when
df = df.withColumn(
    "ChurnFlag",
    when(col("Churn")==1, 1).otherwise(0)
)

In [0]:
# save silver
df.write.format("delta").mode("overwrite").saveAsTable("churn_silver")

In [0]:
# BUSINESS LAYER (Gold) - aggregation
spark.sql("""
CREATE TABLE churn_gold AS
SELECT
    gender,
    AVG(Total_Spend) as avg_monthly,
    AVG(tenure) as avg_tenure,
    SUM(ChurnFlag) as churn_count,
    COUNT(*) as total_customers
FROM churn_silver
GROUP BY gender
    """)

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-6488979240922750>, line 2
      1 # BUSINESS LAYER (Gold) - aggregation
----> 2 spark.sql("""
      3 CREATE TABLE churn_gold AS
      4 SELECT
      5     gender,
      6     AVG(Total_Spend) as avg_monthly,
      7     AVG(tenure) as avg_tenure,
      8     SUM(ChurnFlag) as churn_count,
      9     COUNT(*) as total_customers
     10 FROM churn_silver
     11 GROUP BY gender
     12     """)

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/session.py:901, in SparkSession.sql(self, sqlQuery, args, **kwargs)
    898         _views.append(SubqueryAlias(df._plan, name))
    900 cmd = SQL(sqlQuery, _args, _named_args, _views)
--> 901 data, properties, ei = self.client.execute_command(cmd.command(self._client))
    902 if "sql_command_result" in properties:
    903     df = DataFrame(CachedRelation(pr

In [0]:
# analytics
spark.sql("""
SELECT *, churn_count / total_customers as churn_rate
FROM churn_gold         
""").display()

gender,avg_monthly,avg_tenure,churn_count,total_customers,churn_rate
Male,645.5145085753544,31.37647651167623,122941,250252,0.4912688010485431
Female,613.3662814565987,31.098578024976387,127058,190580,0.6666911533214398


Databricks visualization. Run in Databricks to view.

In [0]:
%sql
SELECT tenure, AVG(ChurnFlag) as churn_rate
FROM churn_silver
GROUP BY tenure;

tenure,churn_rate
39,0.5433205855202426
49,0.5346129238643634
14,0.6382401687509417
38,0.545583861373335
32,0.547138477261114
33,0.5373230373230373
37,0.5402724507340299
12,0.6202321724709784
3,0.6344086021505376
18,0.6449232585596222


Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

In [0]:
# MACHINE LEARNING (ADVANCED)
# Predict Churn
#from pyspark.ml.feature import VectorAssembler
#from pyspark.ml.classification import LogisticRegression

#assembler = VectorAssembler (
#  inputCols=["tenure", "MonthlyCharges"],
#  outputCol="features"
#)
#df_ml = assembler.transform(df)

#model = LogisticRegression(labelCol="ChurnFlag" )
#model_fit = model.fit(df_ml)

In [0]:
# INDUSTRY DASHBOARD:
#   KPI SECTION (TOP ROW)
# KPI 1 : TOTAL CUSTOMERS
df = spark.sql("""
SELECT COUNT(*) as total_customers
FROM churn_silver
""")

df.show()


+---------------+
|total_customers|
+---------------+
|         440832|
+---------------+



In [0]:
# KPI 2: TOTAL CHURN CUSTOMERS
df_churn = spark.sql("""
SELECT SUM(ChurnFlag) as churn_customers
FROM churn_silver                     
""")
df_churn.show()

+---------------+
|churn_customers|
+---------------+
|         249999|
+---------------+



In [0]:
# KPI 3: CHURN RATE
df_rate = spark.sql("""
SELECT ROUND(SUM(ChurnFlag) / COUNT(*) * 100, 2) as churn_rate
FROM churn_silver
""")
df_rate.show()
 

+----------+
|churn_rate|
+----------+
|     56.71|
+----------+



In [0]:
# KPI 4: AVG TOTAL SPEND
df_avg_chargers = spark.sql("""
SELECT ROUND(AVG(Total_Spend), 2) as avg_chargers
FROM churn_silver
""")
df_avg_chargers.show()


+------------+
|avg_chargers|
+------------+
|      631.62|
+------------+



In [0]:
# KPI 5: AVG TENURE
df_avg_tenure = spark.sql("""
SELECT ROUND(AVG(tenure), 2) as avg_tenure
FROM churn_silver                          
""")
df_avg_tenure.show()

+----------+
|avg_tenure|
+----------+
|     31.26|
+----------+



In [0]:
# MAIN CHARTS
# 1. churn trend (line chart)
df_trend = spark.sql("""
SELECT tenure, AVG(ChurnFlag) as churn_rate
FROM churn_silver
GROUP BY tenure
ORDER BY  tenure                   
""")
df_trend.display(df_trend)

tenure,churn_rate
1,0.6536600593101295
2,0.6471482889733841
3,0.6344086021505376
4,0.6413866182258553
5,0.6450742240215924
6,0.5353063343717549
7,0.5457788347205708
8,0.5533246414602346
9,0.5379612423679321
10,0.537529319781079


Databricks visualization. Run in Databricks to view.

In [0]:
# 2.CHURN BY GENDER (BAR CHART):
df_gender = spark.sql("""
SELECT gender, AVG(ChurnFlag) as churn_rate
FROM churn_silver
GROUP BY gender
""")
df_gender.display(df_gender)

gender,churn_rate
Female,0.6666911533214398
Male,0.4912688010485431


Databricks visualization. Run in Databricks to view.

In [0]:
# 3.CHURN BY TENTURE TYPE
df_tenure = spark.sql("""
SELECT Tenure, ROUND(AVG(ChurnFlag)* 100, 2) as churn_rate
FROM churn_silver
GROUP BY Tenure
ORDER BY Tenure
""")
df_tenure.display(df_tenure)

Tenure,churn_rate
1,65.37
2,64.71
3,63.44
4,64.14
5,64.51
6,53.53
7,54.58
8,55.33
9,53.8
10,53.75


Databricks visualization. Run in Databricks to view.

In [0]:
# 4. CHURN BY Subscription_type
df_payment = spark.sql("""
SELECT Subscription_Type, AVG(ChurnFlag) as churn_rate
FROM churn_silver
GROUP BY Subscription_Type                       
""")
df_payment.display(df_payment)

Subscription_Type,churn_rate
Standard,0.5606995332868409
Basic,0.5817823332820606
Premium,0.5594169951169642


Databricks visualization. Run in Databricks to view.

In [0]:
# 5. TOTAL SPEND VS CHURN (Distribution):
df_totalspend = spark.sql("""
SELECT Total_Spend, ChurnFlag
FROM churn_silver                          
""")
df_totalspend.display(df_totalspend)

Total_Spend,ChurnFlag
932.0,1
557.0,1
185.0,1
396.0,1
617.0,1
129.0,1
821.0,1
445.0,1
969.0,1
415.0,1


Databricks visualization. Run in Databricks to view.